# Cat and Dog Image Classifier

Image generators rescale pixels and augment only the training images. A frozen MobileNetV2 convolutional network feeds a trained dense classifier. Validation and test images retain their original order, and the official 50-image check is kept unchanged.

Prepared with AI assistance for an agentic AI class assignment. Starter notebook and original challenge tests: [freeCodeCamp](https://github.com/freeCodeCamp/boilerplate-cat-and-dog-image-classifier). The supplied final test cell is preserved. Run all cells from top to bottom.

In [1]:
import os
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
os.environ.setdefault('TF_NUM_INTRAOP_THREADS', '2')
os.environ.setdefault('TF_NUM_INTEROP_THREADS', '2')
DATA_DIR = Path(os.environ.get('FCC_DATA_DIR', 'data')).resolve()
ARTIFACT_DIR = Path(os.environ.get('FCC_ARTIFACT_DIR', 'artifacts')).resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def data_file(name, relative_url, extract=False):
    # The official freeCodeCamp CDN repository is a public mirror of the datasets.
    path = DATA_DIR / name
    if not path.exists() or not path.stat().st_size:
        url = 'https://raw.githubusercontent.com/freeCodeCamp/cdn/main/build/project-data/' + relative_url
        with urlopen(url, timeout=120) as response:
            contents = response.read()
        temporary = path.with_suffix(path.suffix + '.part')
        temporary.write_bytes(contents)
        temporary.replace(path)
    if extract:
        with ZipFile(path) as archive:
            for member in archive.namelist():
                (DATA_DIR / member).resolve().relative_to(DATA_DIR)
            archive.extractall(DATA_DIR)
    return path

In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Flatten, Dropout, MaxPooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import matplotlib.pyplot as plt
tf.keras.utils.set_random_seed(42)
try:
    tf.config.threading.set_intra_op_parallelism_threads(2)
    tf.config.threading.set_inter_op_parallelism_threads(2)
except RuntimeError:
    pass  # The runtime may already be initialized when a notebook cell is rerun.

def batches(x, y=None, training=False, batch_size=64):
    contents = x if y is None else (x, y)
    ds = tf.data.Dataset.from_tensor_slices(contents)
    if training:
        ds = ds.shuffle(len(x), seed=42)
    options = tf.data.Options()
    options.threading.private_threadpool_size = 2
    options.threading.max_intra_op_parallelism = 2
    return ds.batch(batch_size).with_options(options).prefetch(1)

In [3]:
data_file('cats_and_dogs.zip', 'cats-and-dogs/cats_and_dogs.zip', extract=True)
PATH = str(DATA_DIR / 'cats_and_dogs')
train_dir = os.path.join(PATH, 'train')
validation_dir = os.path.join(PATH, 'validation')
test_dir = os.path.join(PATH, 'test')
total_train = sum(len(files) for _, _, files in os.walk(train_dir))
total_val = sum(len(files) for _, _, files in os.walk(validation_dir))
total_test = len(list(Path(test_dir).glob('*.jpg')))
batch_size = 64
epochs = 5
IMG_HEIGHT = 150
IMG_WIDTH = 150

In [4]:
train_image_generator = ImageDataGenerator(rescale=1./255)
validation_image_generator = ImageDataGenerator(rescale=1./255)
test_image_generator = ImageDataGenerator(rescale=1./255)
train_data_gen = train_image_generator.flow_from_directory(directory=train_dir,
    batch_size=batch_size, target_size=(IMG_HEIGHT, IMG_WIDTH), class_mode='binary', seed=42)
val_data_gen = validation_image_generator.flow_from_directory(directory=validation_dir,
    batch_size=batch_size, target_size=(IMG_HEIGHT, IMG_WIDTH), class_mode='binary', shuffle=False)
test_data_gen = test_image_generator.flow_from_directory(directory=PATH, classes=['test'],
    batch_size=batch_size, target_size=(IMG_HEIGHT, IMG_WIDTH), class_mode=None, shuffle=False)
assert train_data_gen.class_indices == {'cats':0, 'dogs':1}

Found 2000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.
Found 50 images belonging to 1 classes.


In [5]:
def plotImages(images_arr, probabilities=False):
    columns = min(5, len(images_arr))
    rows = int(np.ceil(len(images_arr)/columns))
    fig, axes = plt.subplots(rows, columns, figsize=(3*columns, 3*rows), squeeze=False)
    for index, ax in enumerate(axes.ravel()):
        ax.axis('off')
        if index < len(images_arr):
            ax.imshow(images_arr[index])
            if probabilities is not False:
                probability = float(probabilities[index])
                label = 'dog' if probability >= 0.5 else 'cat'
                confidence = probability if probability >= 0.5 else 1-probability
                ax.set_title(f'{confidence:.1%} {label}')
    plt.tight_layout()
    plt.show()

sample_training_images, _ = next(train_data_gen)
plotImages(sample_training_images[:5])

In [6]:
train_image_generator = ImageDataGenerator(rescale=1./255,
    rotation_range=15, width_shift_range=0.1, height_shift_range=0.1,
    shear_range=0.1, zoom_range=0.15, horizontal_flip=True)

In [7]:
train_data_gen = train_image_generator.flow_from_directory(batch_size=batch_size,
    directory=train_dir, target_size=(IMG_HEIGHT, IMG_WIDTH), class_mode='binary', seed=42)
augmented_images = [train_data_gen[0][0][0] for _ in range(5)]
plotImages(augmented_images)

Found 2000 images belonging to 2 classes.


In [8]:
# Transfer learning uses the pretrained convolutional feature extractor covered in the course.
base_model = keras.applications.MobileNetV2(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    include_top=False, weights='imagenet', pooling='avg')
base_model.trainable = False
model = Sequential([
    keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    keras.layers.Rescaling(2.0, offset=-1.0), base_model,
    Dropout(0.2), Dense(64, activation='relu'), Dropout(0.2),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer=keras.optimizers.Adam(0.0005), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

fcc_cat_dog.ipynb:cell8:2: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
Model: "sequential"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 1280)           │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense) 

In [9]:
def generator_dataset(generator):
    def stream():
        while True:
            for index in range(len(generator)):
                yield generator[index]
            generator.on_epoch_end()
    signature = (tf.TensorSpec((None, IMG_HEIGHT, IMG_WIDTH, 3), tf.float32),
                 tf.TensorSpec((None,), tf.float32))
    ds = tf.data.Dataset.from_generator(stream, output_signature=signature)
    options = tf.data.Options()
    options.threading.private_threadpool_size = 2
    return ds.with_options(options).prefetch(1)

history = model.fit(x=generator_dataset(train_data_gen), steps_per_epoch=len(train_data_gen),
    epochs=epochs, validation_data=generator_dataset(val_data_gen), validation_steps=len(val_data_gen),
    verbose=2, callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=2,
        restore_best_weights=True)])

/workspace/scratch/19945c1fc1e3/ml_runtime/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
Epoch 1/5
32/32 - 18s - 549ms/step - accuracy: 0.8865 - loss: 0.2558 - val_accuracy: 0.9610 - val_loss: 0.0961
Epoch 2/5
32/32 - 14s - 424ms/step - accuracy: 0.9510 - loss: 0.1348 - val_accuracy: 0.9620 - val_loss: 0.0947
Epoch 3/5
32/32 - 14s - 453ms/step - accuracy: 0.9595 - loss: 0.1128 - val_accuracy: 0.9720 - val_loss: 0.0862
Epoch 4/5
32/32 - 14s - 425ms/step - accuracy: 0.9595 - loss: 0.1056 - val_accuracy: 0.9670 - val_loss: 0.0828
Epoch 5/5
32/32 - 14s - 430ms/step - accuracy: 0.9580 - loss: 0.0983 - val_accuracy: 0.9730 - val_loss: 0.0852


In [10]:
# 9
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(len(history.history['accuracy']))

plt.figure(figsize=(8, 8))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

In [11]:
probabilities = np.concatenate([model(test_data_gen[index], training=False).numpy().ravel()
                                for index in range(len(test_data_gen))]).tolist()
test_images = np.concatenate([test_data_gen[index] for index in range(len(test_data_gen))])
assert len(probabilities) == total_test == 50
plotImages(test_images, probabilities)

In [13]:
# 11
answers =  [1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0,
            1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0,
            1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1,
            1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 
            0, 0, 0, 0, 0, 0]

correct = 0

for probability, answer in zip(probabilities, answers):
  if round(probability) == answer:
    correct +=1

percentage_identified = (correct / len(answers)) * 100

passed_challenge = percentage_identified >= 63

print(f"Your model correctly identified {round(percentage_identified, 2)}% of the images of cats and dogs.")

if passed_challenge:
  print("You passed the challenge!")
else:
  print("You haven't passed yet. Your model should identify at least 63% of the images. Keep trying. You will get it!")

Your model correctly identified 94.0% of the images of cats and dogs.
You passed the challenge!


In [14]:
assert passed_challenge, f'Accuracy {percentage_identified:.1f}% does not meet the official requirement.'
model.save(ARTIFACT_DIR / 'cat_dog_classifier.keras')
print(f'PASS: official image accuracy = {percentage_identified:.1f}%')

PASS: official image accuracy = 94.0%
